# D2-03 · One-stage detectors: dense grids, SSD, FPN, RetinaNet, focal loss, anchor-free

### Steps
1. Dense prediction: one output vector per grid cell (the YOLOv1 encoding)
2. Build and train a mini detector end to end on shapes (custom loop, 3-term loss, decode + NMS, mAP)
3. Multi-scale: SSD default boxes and the Feature Pyramid Network
4. RetinaNet: shared heads over the pyramid
5. Class imbalance and focal loss, derived and plotted
6. Anchor-free heads: FCOS, CenterNet, the modern YOLO head
7. Library RetinaNet and FCOS with COCO weights on Pet images

### Topics covered and learning objectives
- Write the target tensor of a dense detector and the loss that trains it
- Train a working detector from scratch in about two minutes on a CPU
- Explain why detectors use feature pyramids and draw an FPN with its tensor shapes
- Derive focal loss from cross-entropy and say what gamma and alpha do
- Compare anchor-based and anchor-free heads

### Time estimate
~90 min in class (guided + hands-on)

### Prerequisites
D2-01 (IoU, NMS, mAP) and D2-02 (anchors, matching, encoding).

## 0 · Setup

<details class="api-ref" data-api="np.set_printoptions,torch.manual_seed,torchkit.get_device"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.set_printoptions</code> · <code>torch.manual_seed</code> · <code>torchkit.get_device</code></summary>
<ul><li><code>np.set_printoptions(precision=2, suppress=True)</code> &nbsp; Changes how NumPy prints arrays for the rest of the session: precision=2 shows two decimals, suppress=True writes tiny numbers as 0. instead of in scientific notation (2e-05). Only the display changes, never the values.</li><li><code>torch.manual_seed(seed)</code> &nbsp; Fixes PyTorch&#x27;s random number generator, so random weights, shuffles and augmentations come out the same on every run. The notebooks call it in the setup cell and again right before creating a model, so that its initial weights are repeatable.</li><li><code>device = torchkit.get_device()</code> &nbsp; Course helper. Returns torch.device(&quot;cuda&quot;) when PyTorch can use an NVIDIA GPU, torch.device(&quot;mps&quot;) on an Apple-silicon Mac, otherwise torch.device(&quot;cpu&quot;), and prints which one it picked (for the CPU, with the number of threads PyTorch uses). Models and batches are then moved there with .to(device).</li></ul></details>

In [ ]:
import sys, pathlib, time
for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:   # find the cvcourse package
    if (p / "cvcourse").is_dir():
        sys.path.insert(0, str(p)); break

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch, Circle
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
from cvcourse import viz, boxes, shapes, tests, torchkit, day2, labs

%matplotlib inline
plt.rcParams["figure.dpi"] = 90
np.set_printoptions(precision=3, suppress=True)
torch.manual_seed(0)
device = torchkit.get_device()
GREEN, RED, GREY, BLUE, ORANGE = "#54A24B", "#E45756", "#9a9a9a", "#4C78A8", "#F58518"

## 1 · Dense prediction on a grid

The fully-convolutional trick of D2-01 gave us a *grid of class scores*. A one-stage detector
makes every grid cell predict the **whole detection**: is there an object here, where exactly
is its box, and which class is it. With $A$ anchors (or "slots") per cell and $C$ classes the
head is one convolution with

$$A \times (\underbrace{1}_{\text{objectness}} + \underbrace{4}_{\text{box}} + \underbrace{C}_{\text{classes}}) \text{ output channels.}$$

No proposals, no RoIAlign, no second stage - one forward pass, then decoding and NMS.

**YOLOv1's assignment rule** (2016): the cell that contains the **centre** of an object is
*responsible* for it; all other cells must predict "no object". We use $A=1$, an 8 x 8 grid
on 128 px shapes images (cell = 16 px), and store each box as $(c_x, c_y, w, h)$ normalised by
the image size.

### Lab · Grid assignment
**Predict first:** two small objects whose box centres fall into the same cell. What does the
target tensor hold for that cell: both boxes, their average, or one of them? And which class?
Drag the shapes to check, then work through the "Try this" list under the lab (5 minutes).

In [ ]:
labs.show("grid-assignment")

### Exercise 1 · the dense target tensor

`grid_targets(boxes, labels, grid, size, num_classes)` -> a float32 array `(grid, grid, 5 + C)`
indexed `[row, col, channel]` (channels last; the training code permutes it to channels-first
later). `boxes` is `(N, 4)` xyxy in pixels, `labels` holds class indices `0 .. C-1`.
Channel 0 objectness, 1-4 normalised `(cx, cy, w, h)`, then a one-hot class.
The responsible cell is `(row, col) = (int(cy // cell), int(cx // cell))`.
These are exactly the 8 numbers per object in the table under the lab.

<details class="api-ref" data-api="np.zeros"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.zeros</code></summary>
<ul><li><code>np.zeros(shape, dtype=float)</code> &nbsp; A new array of the given shape filled with 0.0 (float64). The standard way to allocate an output array before filling it in a loop.</li></ul></details>

In [ ]:
def grid_targets(bxs, labels, grid=8, size=128, num_classes=3):
    '''bxs (N, 4) xyxy in pixels, labels (N,) class indices -> T (grid, grid, 5 + num_classes) float32,
    indexed [row, col, channel]: 0 objectness, 1-4 (cx, cy, w, h) / size, then a one-hot class.'''
    T = np.zeros((grid, grid, 5 + num_classes), np.float32)
    cell = size / grid                               # side of one cell in pixels (16 for 128 / 8)
    for box, label in zip(bxs, labels):
        # 1. the centre of the box, in pixels
        cx = (box[0] + box[2]) / 2
        cy = (box[1] + box[3]) / 2
        # 2. the responsible cell
        i = int(cy // cell)                          # row
        j = int(cx // cell)                          # column
        # 3. objectness
        T[i, j, 0] = 1
        # 4. the box, normalised by the image size
        w = box[2] - box[0]
        h = box[3] - box[1]
        T[i, j, 1:5] = [cx / size, cy / size, w / size, h / size]
        # 5. the one-hot class
        T[i, j, 5 + int(label)] = 1
    return T

tests.check_grid_targets(grid_targets)

<details class="api-ref" data-api="np.ndenumerate"><summary><b>Functions used here</b> (click to open): <code>np.ndenumerate</code></summary>
<p><b><code>for index, value in np.ndenumerate(a):</code></b><br>Loops over every element of an array of any shape, giving its index tuple together with its value (enumerate for N-D arrays). D2-03 uses it to write each grid cell&#x27;s value on a plot.</p><pre>&gt;&gt;&gt; for (i, j), v in np.ndenumerate(np.array([[1, 2], [3, 4]])):
...     print(i, j, v)
0 0 1
0 1 2
1 0 3
1 1 4</pre><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.random.default_rng"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.random.default_rng</code></summary>
<ul><li><code>rng = np.random.default_rng(seed)</code> &nbsp; Creates a NumPy random number generator; the same seed gives the same numbers on every run. Draw from it with rng.normal(size=...) (normal distribution), rng.random(size) (uniform in [0, 1)), rng.integers(low, high, size) or rng.permutation(n).</li></ul></details>

In [ ]:
g_img, g_boxes, g_labels, _ = shapes.make_sample(np.random.default_rng(7), 128)
T = shapes.grid_targets(g_boxes, g_labels, 8, 128, 3)
fig = plt.figure(figsize=(16, 5.2))
ax = fig.add_axes([0.0, 0.05, 0.33, 0.85]); ax.imshow(g_img)
for k in range(9):
    ax.axhline(k * 16 - 0.5, color="k", lw=0.5, alpha=0.5); ax.axvline(k * 16 - 0.5, color="k", lw=0.5, alpha=0.5)
for b, l in zip(g_boxes, g_labels):
    cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2; i, j = int(cy // 16), int(cx // 16)
    ax.add_patch(Rectangle((j * 16 - 0.5, i * 16 - 0.5), 16, 16, fc="yellow", alpha=0.45, ec="k", lw=2))
    ax.plot(cx, cy, "k+", ms=12, mew=2)
    ax.add_patch(Rectangle(b[:2], *(b[2:] - b[:2]), fill=False, ec="k", lw=1.2, ls="--"))
ax.set_title("8 x 8 grid: the cell holding each centre (+) is responsible"); ax.axis("off")
names = ["objectness", "cx", "cy", "w", "h"] + [f"class: {n}" for n in shapes.CLASS_NAMES]
for c in range(8):
    a = fig.add_axes([0.35 + (c % 4) * 0.165, 0.5 - (c // 4) * 0.47, 0.15, 0.4])
    a.imshow(T[..., c], cmap="viridis" if 1 <= c <= 4 else "Greys", vmin=0, vmax=1)
    for (i, j), v in np.ndenumerate(T[..., c]):
        if T[i, j, 0] > 0:
            a.text(j, i, f"{v:.2f}" if 1 <= c <= 4 else f"{v:.0f}", ha="center", va="center", fontsize=7, color="red")
    a.set_title(f"channel {c}: {names[c]}", fontsize=9); a.set_xticks([]); a.set_yticks([])
fig.suptitle("The target tensor (8, 8, 5 + 3): only responsible cells carry a box and a class", y=1.0)
plt.show()

What to look at: every channel is zero except in the four responsible (yellow) cells, and the box
channels hold numbers relative to the *whole image*: the circle on the right has `cx = 0.84`
whatever cell it sits in. The other 60 cells only teach "no object".

## 2 · A mini detector, trained end to end

`day2.MiniDetector`: four stride-2 stages (128 -> 64 -> 32 -> 16 -> 8, conv-BN-ReLU) and a
1 x 1 head with 5 + 3 channels. To make the box easy to learn, the model **decodes inside
`forward`**: for the cell in row $i$, column $j$ of a $G \times G$ grid

$$c_x = \frac{j + \sigma(t_x)}{G}, \quad c_y = \frac{i + \sigma(t_y)}{G}, \quad w = \sigma(t_w), \quad h = \sigma(t_h),$$

so a cell can only place a centre *inside itself* (exactly YOLO's parametrisation) and the
output already lives in the same layout as the target. Channel 0 and the class channels stay
logits for numerically stable losses.

<details class="api-ref" data-api="torchkit.summarize"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.summarize</code></summary>
<ul><li><code>print(torchkit.summarize(model, input_size))</code> &nbsp; Course helper. Pushes one dummy (all-zero) image of input_size, (C, H, W), through the model and returns a table as a string: one row per layer with its name, type, output shape, number of parameters and MACs (multiply-accumulates, counted for conv and linear layers), then the totals. The model&#x27;s train / eval mode is restored afterwards.</li></ul></details>

In [ ]:
print(torchkit.summarize(day2.MiniDetector(), (3, 128, 128)))

### Exercise 2 · the three-term detection loss

`pred` and `target` are `(N, 5 + C, G, G)`. With `pos = target[:, 0] > 0.5` (responsible cells):

* **objectness**: `F.binary_cross_entropy_with_logits` over **all** cells (mean) - every cell must learn yes / no;
* **box**: L1 between `pred[:, 1:5]` and `target[:, 1:5]`, summed over the 4 numbers, averaged over **positive cells only**;
* **class**: `F.cross_entropy` on the class logits of **positive cells only** (target = argmax of the one-hot);
* total = `1 * obj + 5 * box + 1 * cls`. Return `(total, {"obj": ..., "box": ..., "cls": ...})`.

Tip: `pred[:, 1:5].permute(0, 2, 3, 1)[pos]` gives a `(P, 4)` tensor of the positive cells.
Guard the case "no object in the batch" (box and class loss = 0, not NaN).

<details class="api-ref" data-api="F.binary_cross_entropy_with_logits"><summary><b>Functions you will need</b> (click to open): <code>F.binary_cross_entropy_with_logits</code></summary>
<p><b><code>F.binary_cross_entropy_with_logits(logits, targets, reduction=&#x27;mean&#x27;)</code></b><br>Binary cross-entropy for yes / no outputs, computed from raw logits z: per element -[t log sigmoid(z) + (1 - t) log(1 - sigmoid(z))], then the mean (reduction=&quot;none&quot; keeps one loss per element, &quot;sum&quot; adds them up). targets are floats 0. / 1. of the same shape. Objectness and focal loss use it.</p><pre>&gt;&gt;&gt; logits = torch.tensor([2.0, -1.0, 0.0])
&gt;&gt;&gt; targets = torch.tensor([1.0, 0.0, 1.0])
&gt;&gt;&gt; F.binary_cross_entropy_with_logits(logits, targets, reduction=&quot;none&quot;)
tensor([0.1269, 0.3133, 0.6931])
&gt;&gt;&gt; F.binary_cross_entropy_with_logits(logits, targets)          # the mean
tensor(0.3778)</pre><p><i>Watch out:</i> Give it logits, not probabilities (it applies the sigmoid itself, in a numerically stable way), and float targets: integer targets raise an error.</p><p>See also: <code>torch.sigmoid</code>, <code>F.binary_cross_entropy</code>, <code>F.cross_entropy</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="F.cross_entropy,Tensor.permute,Tensor.any,Tensor.abs,Tensor.sum,Tensor.mean,Tensor.argmax"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>F.cross_entropy</code> · <code>Tensor.permute</code> · <code>Tensor.any</code> · <code>Tensor.abs</code> · <code>Tensor.sum</code> · <code>Tensor.mean</code> · <code>Tensor.argmax</code></summary>
<ul><li><code>F.cross_entropy(logits, target, label_smoothing=0.0)</code> &nbsp; The classification loss: logits (B, C) are raw scores, target is either the class indices (B,) as int64 or a (B, C) float tensor of probabilities (soft targets, e.g. from Mixup or CutMix). It applies log_softmax itself and returns the mean of -log p(correct class) over the batch. label_smoothing=eps trains against (1 - eps) on the true class plus eps / C spread over all classes.</li><li><code>x.permute(*dims)</code> &nbsp; Reorders the dimensions: the argument lists the old positions in their new order. No data is copied. Matplotlib and image files use (H, W, C); PyTorch layers use (C, H, W); permute converts.</li><li><code>x.any(dim=None)</code> &nbsp; True if at least one element is True (non-zero): over the whole tensor, or along dim (one answer per row with dim=1). Top-k accuracy: compare the k guesses of each image with its label, then ask whether any of them matched.</li><li><code>x.abs()</code> &nbsp; The absolute value of every element. (a - b).abs().max() is the largest difference between two tensors: the usual check that two results agree.</li><li><code>x.sum(dim=None, keepdim=False)</code> &nbsp; Adds up all elements (giving a 0-dimensional tensor), or only along dim, which then disappears from the shape: for a (2, 3) tensor, sum(dim=0) adds the two rows together and gives one value per column, shape (3,). dim can be a tuple; keepdim=True keeps the summed dimension with size 1.</li><li><code>x.mean(dim=None, keepdim=False)</code> &nbsp; The average of all elements, or along dim (a number or a tuple), which then disappears from the shape. For a batch (B, C, H, W), mean(dim=(2, 3)) averages over the pixels and gives (B, C): global average pooling. NumPy arrays have the same method, with axis= for dim=.</li><li><code>x.argmax(dim)</code> &nbsp; The position of the largest value along dim. For logits (B, num_classes), x.argmax(dim=1) is the predicted class of every sample, shape (B,). Without dim it gives the position in the flattened tensor.</li></ul></details>

In [ ]:
def dense_loss(pred, target, w_obj=1.0, w_box=5.0, w_cls=1.0):
    '''pred, target: (N, 5 + C, G, G); channel 0 objectness, 1-4 the box, 5.. the classes
    (pred holds logits for objectness and classes, target a 0/1 objectness and a one-hot class)
    -> (total, {"obj": l_obj, "box": l_box, "cls": l_cls}).'''
    # 1. the responsible cells
    pos = target[:, 0] > 0.5                                  # (N, G, G) True / False
    # 2. objectness: binary cross-entropy on every cell
    l_obj = F.binary_cross_entropy_with_logits(pred[:, 0], target[:, 0])
    if pos.any():
        # 3. box: channels last, so that [pos] picks one row of 4 numbers per positive cell
        pred_box = pred[:, 1:5].permute(0, 2, 3, 1)           # (N, G, G, 4)
        pred_box = pred_box[pos]                              # (P, 4), P = number of positive cells
        target_box = target[:, 1:5].permute(0, 2, 3, 1)       # (N, G, G, 4)
        target_box = target_box[pos]                          # (P, 4)
        abs_err = (pred_box - target_box).abs()               # (P, 4)
        l1_per_cell = abs_err.sum(-1)                         # (P,) summed over the 4 numbers
        l_box = l1_per_cell.mean()                            # mean over the positive cells
        # 4. class: cross-entropy on the positive cells
        logits = pred[:, 5:].permute(0, 2, 3, 1)              # (N, G, G, C)
        logits = logits[pos]                                  # (P, C)
        target_onehot = target[:, 5:].permute(0, 2, 3, 1)     # (N, G, G, C)
        target_onehot = target_onehot[pos]                    # (P, C)
        target_class = target_onehot.argmax(-1)               # (P,) class index of every positive cell
        l_cls = F.cross_entropy(logits, target_class)
    else:
        # no object in the batch: box and class loss 0, not NaN (pred.sum() * 0.0 is a 0 that is still a tensor)
        l_box = pred.sum() * 0.0
        l_cls = pred.sum() * 0.0
    # 5. the weighted sum
    total = w_obj * l_obj + w_box * l_box + w_cls * l_cls
    return total, {"obj": l_obj, "box": l_box, "cls": l_cls}

tests.check_dense_loss(dense_loss)

### Train (about 2 minutes on a laptop CPU)

1500 training images, 6 epochs, AdamW with a one-cycle schedule, random horizontal flips
(the flip must also flip the grid columns and $c_x$ - look at `day2.train_mini_detector`, it is
20 lines). The loop uses **your** loss if it passed the checker. A time cap stops training
after 3 minutes on a very slow machine (even 2 epochs reach mAP@0.5 > 0.95).

> **CPU tip.** Tiny models like this one are dominated by thread-synchronisation overhead: on a
> busy laptop all cores can be several times *slower* than 1 or 2 threads. The training loop
> therefore first times one batch with a few thread counts and uses the fastest (first line of
> the log). Benchmark, do not guess.

In [ ]:
loss_fn = dense_loss if tests.check_dense_loss(dense_loss, verbose=False) else day2.dense_loss
print("using", "your dense_loss" if loss_fn is dense_loss else "the reference day2.dense_loss")
X_tr, B_tr, L_tr = day2.shapes_tensors(1500, 128, seed=0)
T_tr = day2.grid_target_tensor(B_tr, L_tr)
X_va, B_va, L_va = day2.shapes_tensors(300, 128, seed=1)
print("train", tuple(X_tr.shape), "targets", tuple(T_tr.shape))

torch.manual_seed(0)
det = day2.MiniDetector()
t0 = time.time()
hist = day2.train_mini_detector(det, X_tr, T_tr, loss_fn=loss_fn, epochs=6, batch_size=32, lr=4e-3, max_seconds=180)
train_seconds = time.time() - t0
fig, axes = plt.subplots(1, 4, figsize=(16, 3.2))
for ax, k, c in zip(axes, ["total", "obj", "box", "cls"], ["k", ORANGE, BLUE, GREEN]):
    ax.plot(range(1, len(hist[k]) + 1), hist[k], "o-", color=c); ax.set_title(f"{k} loss"); ax.set_xlabel("epoch"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

What to look at: the class loss is close to 0 after two epochs (telling circles from squares is
easy), while the box loss keeps falling until the last epoch: placing a box to the pixel is the
hard part of detection.

### Exercise 3 · decode the grid

Inverse of Exercise 1 for a prediction `(G, G, 5 + C)` whose objectness and classes are already
probabilities: every cell with objectness $\ge$ threshold becomes a box (scale `cx, cy, w, h` by
`size`, convert to xyxy), its score is the objectness, its label the argmax of the class part.
`day2.decode_predictions` then multiplies score by class probability and runs class-wise NMS.

<details class="api-ref" data-api="np.array,Tensor.reshape,np.argmax,np.arange,Tensor.round"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.array</code> · <code>Tensor.reshape</code> · <code>np.argmax</code> · <code>np.arange</code> · <code>Tensor.round</code></summary>
<ul><li><code>np.array(data, dtype=None)</code> &nbsp; Creates a NumPy array from a (nested) list, copying the data. The dtype is inferred: integers give int64, any float gives float64; dtype=float gives floats from integer literals.</li><li><code>x.reshape(*shape)</code> &nbsp; Returns the same elements in a new shape, read in row order (the last dimension changes fastest). One size may be -1: it is computed from the others. NumPy arrays have the same method.</li><li><code>np.argmax(a, axis=None)</code> &nbsp; Index of the largest value: of the flattened array, or one per row / column along axis. On a boolean array it is the index of the first True, e.g. the first step at which IoU reached 0.5.</li><li><code>np.arange(stop) / np.arange(start, stop, step, dtype=None)</code> &nbsp; Like Python&#x27;s range, as a NumPy array: from start (default 0) up to but not including stop. With reshape, the quickest way to build a small test array with distinct values.</li><li><code>x.round(decimals=0)</code> &nbsp; Rounds every element to the nearest integer, or to decimals digits after the point; exact halves go to the even neighbour (0.5 -&gt; 0, 1.5 -&gt; 2). NumPy arrays have the same method: the course uses arr.round(2) to print short numbers.</li></ul></details>

In [ ]:
def decode_grid(T, size=128, score_threshold=0.5):
    '''T (G, G, 5 + C): channel 0 objectness, 1-4 (cx, cy, w, h) / size, 5.. class probabilities ->
    (boxes (K, 4) xyxy in pixels, scores (K,), labels (K,)): one box per cell with objectness >= score_threshold.'''
    bxs = []
    scores = []
    labels = []
    G = T.shape[0]
    for i in range(G):                               # row
        for j in range(G):                           # column
            p = T[i, j]                              # (5 + C,) the numbers of this cell
            # 1. skip the cells without an object
            if p[0] < score_threshold:
                continue
            # 2. the box in pixels
            cx = p[1] * size
            cy = p[2] * size
            w = p[3] * size
            h = p[4] * size
            # 3. as xyxy
            bxs.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
            # 4. the score
            scores.append(float(p[0]))
            # 5. the label: the class with the largest probability
            labels.append(int(np.argmax(p[5:])))
    bxs = np.array(bxs).reshape(-1, 4)               # (K, 4); (0, 4) when no cell passed
    return bxs, np.array(scores), np.array(labels, int)

tests.check_decode_grid(decode_grid)

In [ ]:
t0 = time.time()
P_va = day2.predict_batches(det, X_va)                          # (300, 8, 8, 8) raw outputs
preds = day2.decode_predictions(P_va, 128, score_threshold=0.05, iou_threshold=0.5)
gts = day2.gt_dicts(B_va, L_va)
map50, per_class = boxes.mean_average_precision(preds, gts, 3, (0.5,))
map_coco, _ = boxes.mean_average_precision(preds, gts, 3, tuple(np.arange(0.5, 0.96, 0.05)))
print(f"mini detector on 300 validation images: mAP@0.5 = {map50:.3f}   mAP@[.5:.95] = {map_coco:.3f}")
print("AP@0.5 per class:", dict(zip(shapes.CLASS_NAMES, per_class.round(3))), f"| trained {train_seconds:.0f} s, eval {time.time() - t0:.1f} s")

probs = day2.grid_to_probs(P_va)
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for c, i in enumerate(range(6)):
    axes[0, c].imshow(X_va[i].permute(1, 2, 0)); axes[0, c].axis("off")
    viz.draw_boxes(axes[0, c], B_va[i], edgecolor="k", lw=2.5, alpha=0.4)
    k = preds[i]["scores"] > 0.3
    day2.draw_detections(axes[0, c], preds[i]["boxes"][k], preds[i]["labels"][k], preds[i]["scores"][k], shapes.CLASS_NAMES)
    axes[1, c].imshow(probs[i, ..., 0], cmap="magma", vmin=0, vmax=1); axes[1, c].set_xticks([]); axes[1, c].set_yticks([])
    axes[1, c].set_title("objectness (8 x 8)", fontsize=9)
fig.suptitle("predictions (coloured, score > 0.3) vs ground truth (black) and the objectness grid", y=1.0)
plt.tight_layout(); plt.show()

In the objectness grids (bottom row), some objects light up two neighbouring cells: their centre
lies near a cell border, and two cells that see almost the same pixels are both fairly confident
(the border jump of the lab's "Try this" 3). Class-wise NMS then keeps one of the two boxes.

Expected: **mAP@0.5 above 0.95** after ~6 epochs (well above the 0.9 target), and mAP@[.5:.95]
around 0.8: the grid detector finds everything, its boxes are good but not pixel-perfect.
That is a complete detector: ~300 k parameters, one forward pass, decode, NMS.

What breaks YOLOv1 on real images: **one object per cell** (two small birds in one cell - one is
lost), **one scale** (a 16 px cell must handle a 16 px and a 500 px object), and objects whose
centre falls near a cell border flip between cells. The rest of this notebook fixes these.

## 3 · Multi-scale: SSD default boxes and the Feature Pyramid Network

**SSD** (2016) attaches a detection head to **several feature maps** of decreasing resolution:
fine maps (stride 8) handle small objects, coarse maps (stride 64+) large ones, each with a few
"default boxes" (SSD's name for anchors) whose scale grows with depth.

<details class="api-ref" data-api="torchkit.get_pet_datasets,np.asarray,np.linspace,np.sqrt"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchkit.get_pet_datasets</code> · <code>np.asarray</code> · <code>np.linspace</code> · <code>np.sqrt</code></summary>
<ul><li><code>train, val, classes = torchkit.get_pet_datasets(size=224, target=&quot;category&quot;, subset=None)</code> &nbsp; Course helper. Returns (train, val, classes) for Oxford-IIIT Pet (run tools/download_data.py first); images are float (3, size, size), resized and normalised with the ImageNet mean and std. target chooses the label: &quot;category&quot; gives (image, breed index 0..36) and the 37 breed names; &quot;segmentation&quot; gives (image, int64 mask (H, W)) with 0 background, 1 pet, 2 border; &quot;detection&quot; gives (image, {&quot;boxes&quot;: (N, 4) as x1, y1, x2, y2 in pixels, &quot;labels&quot;: (N,)}) for the pet&#x27;s head. subset=N keeps a class-balanced N training (and N // 4 validation) images for quick experiments.</li><li><code>np.asarray(obj)</code> &nbsp; Converts its argument (a list, a PIL image, a CPU tensor, ...) to a NumPy array; an array is returned as it is, without a copy. A grey PIL image becomes a (H, W) uint8 array, a colour one (H, W, 3).</li><li><code>np.linspace(start, stop, num=50)</code> &nbsp; num evenly spaced numbers from start to stop, both ends included. Used for plot axes and grids.</li><li><code>np.sqrt(x)</code> &nbsp; Square root of every element. Anchors use it to change the shape at a fixed area: for scale s and aspect ratio r = h / w, w = s / sqrt(r) and h = s * sqrt(r), so w * h = s^2 for every ratio.</li></ul></details>

In [ ]:
from PIL import Image
pet_train, pet_val, _ = torchkit.get_pet_datasets(target="detection")
ssd_img = np.asarray(Image.open(pet_val.image_path(12)).convert("RGB").resize((300, 300))) / 255.0
maps = [38, 19, 10, 5, 3, 1]
s_k = np.linspace(0.2, 0.9, 6)                                  # SSD scales (fraction of the image)
fig, axes = plt.subplots(1, 6, figsize=(17, 3.4))
for ax, f, s in zip(axes, maps, s_k):
    ax.imshow(ssd_img); step = 300 / f
    for k in range(f + 1):
        ax.axhline(k * step, color="w", lw=0.4, alpha=0.7); ax.axvline(k * step, color="w", lw=0.4, alpha=0.7)
    c = (f // 2 + 0.5) * step
    for r, col in zip([1, 2, 0.5], [RED, ORANGE, BLUE]):
        w, h = 300 * s / np.sqrt(r), 300 * s * np.sqrt(r)
        ax.add_patch(Rectangle((c - w / 2, c - h / 2), w, h, fill=False, ec=col, lw=1.8))
    ax.plot(c, c, "y*", ms=10, mec="k")
    ax.set_xlim(-60, 360); ax.set_ylim(360, -60); ax.axis("off")
    ax.set_title(f"{f} x {f} map, scale {s:.2f}", fontsize=10)
fig.suptitle("SSD300: default boxes at one location of each of the 6 maps (8732 boxes in total)", y=1.02)
plt.tight_layout(); plt.show()

What to look at: from left to right the cells grow from 8 px to the whole image and the default
boxes from 20% to 90% of the image, so each map is responsible for one band of object sizes.

SSD's problem: the high-resolution maps come from **early** layers, which see edges, not
objects - so small-object detection stayed weak. The **Feature Pyramid Network** (Lin et al.
2017) gives every level both resolution *and* semantics:

1. **bottom-up**: the backbone's stage outputs $C_3, C_4, C_5$ (strides 8, 16, 32);
2. **lateral** 1 x 1 convs bring each to the same width $d$ (256);
3. **top-down**: start at the coarsest, upsample x2 (nearest) and **add** to the next finer lateral;
4. a 3 x 3 conv smooths each merged map (removes upsampling aliasing) -> $P_3, P_4, P_5$;
5. RetinaNet adds $P_6, P_7$ with stride-2 convs for very large objects.

Every level now carries the "what" of $C_5$ at its own resolution.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6.4)); ax.axis("off"); ax.set_xlim(0, 14); ax.set_ylim(-0.7, 6.2)
C = [("C3", "128 x 32 x 32", 2.2), ("C4", "256 x 16 x 16", 1.6), ("C5", "512 x 8 x 8", 1.1)]
P = [("P3", "256 x 32 x 32", 2.2), ("P4", "256 x 16 x 16", 1.6), ("P5", "256 x 8 x 8", 1.1)]
ys = [0.4, 2.4, 4.2]
ax.add_patch(FancyBboxPatch((1.2, -0.55), 1.4, 0.35, boxstyle="round,pad=0.02", fc="#eee", ec="k"))
ax.text(2.8, -0.38, "input 3 x 256 x 256 (ResNet-18)", va="center", fontsize=9)
ax.annotate("", (1.9, 0.4), (1.9, -0.18), arrowprops=dict(arrowstyle="-|>", lw=1.2))
for (n, s, w), y in zip(C, ys):
    ax.add_patch(FancyBboxPatch((1.6 - w / 2 + 0.3, y), w, 1.2, boxstyle="round,pad=0.02", fc="#c6dbef", ec="k"))
    ax.text(1.9, y + 0.6, f"{n}\n{s}", ha="center", va="center", fontsize=9)
for (n, s, w), y in zip(P, ys):
    ax.add_patch(FancyBboxPatch((8.0 - w / 2, y), w, 1.2, boxstyle="round,pad=0.02", fc="#c7e9c0", ec="k"))
    ax.text(8.0, y + 0.6, f"{n}\n{s}", ha="center", va="center", fontsize=9)
    ax.annotate("", (8.0 - w / 2 - 0.05, y + 0.6), (3.1, y + 0.6), arrowprops=dict(arrowstyle="-|>", lw=1.5))
    ax.text(3.2, y + 0.72, "1x1 conv (lateral) -> 256", fontsize=8)
    ax.add_patch(Circle((6.2, y + 0.6), 0.18, fc="white", ec="k")); ax.text(6.2, y + 0.6, "+", ha="center", va="center")
    ax.text(6.45, y + 0.72, "3x3 conv", fontsize=8)
for y0, y1 in [(4.2, 2.4 + 1.2), (2.4, 0.4 + 1.2)]:
    ax.annotate("", (6.2, y1 - 0.42), (6.2, y0 + 0.42), arrowprops=dict(arrowstyle="-|>", lw=1.5, color=RED))
    ax.text(6.35, (y0 + y1) / 2 - 0.1, "upsample x2", color=RED, fontsize=8)
for y0, y1 in [(0.4 + 1.2, 2.4), (2.4 + 1.2, 4.2)]:
    ax.annotate("", (1.9, y1), (1.9, y0), arrowprops=dict(arrowstyle="-|>", lw=1.2))
ax.text(1.9, 5.7, "bottom-up\n(backbone)", ha="center", fontsize=10, weight="bold")
ax.text(6.2, 5.7, "top-down + lateral", ha="center", fontsize=10, weight="bold", color=RED)
ax.text(8.0, 5.7, "pyramid", ha="center", fontsize=10, weight="bold", color=GREEN)
ax.text(11.0, 3.1, "the SAME head runs on every P level\n(RetinaNet, FCOS, Faster R-CNN's RPN)", fontsize=10, va="center")
plt.show()

### Lab · FPN levels
**Predict first:** a 16 x 16 px object in a 512 px image. What fraction of one $P_5$ cell (stride 32)
does it cover, and which level will RetinaNet's anchors assign it to? Use the size presets to
check, then open the "top-down pathway" view and predict the shapes before you write the
`FPN` below (6 minutes).

In [ ]:
labs.show("fpn-levels")

### Exercise 4 · `FPN`

`FPN(in_channels_list, out_channels)` with `nn.ModuleList`s `lateral` (1x1) and `smooth` (3x3,
padding 1). `forward(feats)` takes the list `[C3, C4, C5]` (finest first) and returns
`[P3, P4, P5]`. Upsample with `F.interpolate(top, size=lat.shape[-2:], mode="nearest")`.
Every $P_i$ has `out_channels` channels and the height and width of its $C_i$: the shapes of the
lab's pathway view.

<details class="api-ref" data-api="nn.ModuleList"><summary><b>Functions you will need</b> (click to open): <code>nn.ModuleList</code></summary>
<p><b><code>nn.ModuleList([module, ...])</code></b><br>A list of layers that the model knows about: their parameters appear in model.parameters() (so the optimizer trains them) and move with model.to(device). Use it for a variable number of layers that your forward() calls in its own way, e.g. one lateral conv per pyramid level; unlike nn.Sequential it has no forward of its own.</p><pre>&gt;&gt;&gt; class Lateral(nn.Module):
...     def __init__(self):
...         super().__init__()
...         self.convs = nn.ModuleList([nn.Conv2d(c, 16, 1) for c in (32, 64)])
&gt;&gt;&gt; m = Lateral()
&gt;&gt;&gt; len(m.convs), m.convs[1].in_channels
(2, 64)
&gt;&gt;&gt; sum(p.numel() for p in m.parameters())      # (32 * 16 + 16) + (64 * 16 + 16)
1568</pre><p><i>Watch out:</i> A plain Python list of layers looks the same but is invisible to the model: not in parameters(), not trained, not moved to the GPU.</p><p>See also: <code>nn.Sequential</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="nn.Conv2d,F.interpolate"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>nn.Conv2d</code> · <code>F.interpolate</code></summary>
<ul><li><code>nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, bias=True)</code> &nbsp; A convolution layer: out_channels learned kernels of size (in_channels, k, k), each slid over the input exactly like this notebook&#x27;s cross-correlation (no kernel flip). Input (B, C_in, H, W) -&gt; output (B, C_out, H_out, W_out), with H_out from the output-size formula. The weight has shape (C_out, C_in, k, k), the bias (C_out,).</li><li><code>F.interpolate(x, size=None, scale_factor=None, mode=&quot;nearest&quot;, align_corners=None)</code> &nbsp; Resizes a batch of maps (N, C, H, W) to size=(h, w), or by scale_factor. mode=&quot;nearest&quot; repeats pixels, mode=&quot;bilinear&quot; blends neighbouring pixels for a smooth result (with align_corners=False, the usual convention for images). E.g. to upsample a coarse 5 x 5 or 7 x 7 Grad-CAM map to the image size.</li></ul></details>

In [ ]:
class FPN(nn.Module):
    '''FPN(in_channels_list, d): forward([C3, C4, C5], finest first, C_i of shape (N, c_i, H_i, W_i))
    -> [P3, P4, P5]: P_i of shape (N, d, H_i, W_i).'''
    def __init__(self, in_channels_list, out_channels=256):
        super().__init__()
        # 1. one lateral 1x1 conv per level, c_i channels -> out_channels
        self.lateral = nn.ModuleList()                   # a list of layers that PyTorch knows about
        for c in in_channels_list:
            self.lateral.append(nn.Conv2d(c, out_channels, 1))
        # 2. one smoothing 3x3 conv per level, out_channels -> out_channels (padding 1 keeps the size)
        self.smooth = nn.ModuleList()
        for level in range(len(in_channels_list)):
            self.smooth.append(nn.Conv2d(out_channels, out_channels, 3, padding=1))

    def forward(self, feats):
        # 3. lat: the lateral conv of every level applied to its feature map
        lat = []
        for i in range(len(feats)):
            lat.append(self.lateral[i](feats[i]))        # (N, d, H_i, W_i)
        outs = [None] * len(feats)                       # P3, P4, P5, filled from the coarsest level down
        # 4. the coarsest level: only smoothed
        top = lat[-1]
        outs[-1] = self.smooth[-1](top)
        # 5. top-down: from the second-coarsest level to the finest
        for i in range(len(feats) - 2, -1, -1):          # i = len - 2, ..., 1, 0
            upsampled = F.interpolate(top, size=lat[i].shape[-2:], mode="nearest")   # (N, d, H_i, W_i)
            top = lat[i] + upsampled                     # the merge
            outs[i] = self.smooth[i](top)                # (N, d, H_i, W_i)
        return outs

tests.check_fpn_shapes(FPN)

<details class="api-ref" data-api="torchvision.models.feature_extraction.create_feature_extractor"><summary><b>Functions used here</b> (click to open): <code>torchvision.models.feature_extraction.create_feature_extractor</code></summary>
<p><b><code>create_feature_extractor(model, return_nodes={&#x27;layer2&#x27;: &#x27;C3&#x27;, ...})</code></b><br>Wraps a model so that calling it returns a dict of intermediate outputs instead of the final one: return_nodes maps names of internal layers (&quot;layer2&quot;, &quot;layer3&quot;, ... for a ResNet) to the keys you want. The usual way to get the C3 / C4 / C5 maps of a backbone for an FPN.</p><pre>&gt;&gt;&gt; from torchvision.models.feature_extraction import create_feature_extractor
&gt;&gt;&gt; body = create_feature_extractor(torchvision.models.resnet18(weights=None), {&quot;layer2&quot;: &quot;C3&quot;, &quot;layer4&quot;: &quot;C5&quot;})
&gt;&gt;&gt; out = body(torch.zeros(1, 3, 64, 64))
&gt;&gt;&gt; {k: tuple(v.shape) for k, v in out.items()}       # strides 8 and 32
{&#x27;C3&#x27;: (1, 128, 8, 8), &#x27;C5&#x27;: (1, 512, 2, 2)}</pre><p><i>Watch out:</i> Valid names are listed by torchvision.models.feature_extraction.get_graph_node_names(model); a misspelt name raises an error.</p><p>See also: <code>Module.register_forward_hook</code>, <code>Module.children</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torchvision.models.resnet18,Module.eval,torch.no_grad,torch.randn,torchkit.count_parameters"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchvision.models.resnet18</code> · <code>Module.eval</code> · <code>torch.no_grad</code> · <code>torch.randn</code> · <code>torchkit.count_parameters</code></summary>
<ul><li><code>torchvision.models.resnet18(weights=None, num_classes=1000)</code> &nbsp; Builds a ResNet-18: a stem (conv1 7x7 stride 2, bn1, relu, maxpool), four stages layer1 .. layer4 of two basic residual blocks each (64, 128, 256, 512 channels), global average pooling (avgpool) and a linear head fc (512 -&gt; num_classes); 11.7 M parameters. weights=None gives random weights and downloads nothing; weights=&quot;DEFAULT&quot; downloads the ImageNet weights. Thanks to the global pooling it accepts other input sizes than 224 px. Every torchvision model works this way: resnet50, mobilenet_v3_large, efficientnet_b0, convnext_tiny, ...</li><li><code>model.eval()</code> &nbsp; Puts the model and all its layers into evaluation mode: dropout is switched off and BatchNorm uses the running averages collected during training, so a prediction no longer depends on the rest of the batch. Returns the model, so get_model(...).eval() works in one line.</li><li><code>with torch.no_grad(): ...</code> &nbsp; A with block in which autograd records nothing: results have requires_grad=False, and computing them takes less memory and time. Use it when you only evaluate a model (testing, plotting its predictions), never around the training step itself.</li><li><code>torch.randn(*size, generator=None)</code> &nbsp; A tensor of the given shape filled with random numbers from the standard normal distribution (mean 0, standard deviation 1); 0.4 * torch.randn(n) has standard deviation 0.4. The course uses it for noise and for random test inputs.</li><li><code>torchkit.count_parameters(model, trainable_only=False)</code> &nbsp; Course helper. The total number of parameters of a model, sum(p.numel() for p in model.parameters()); trainable_only=True counts only those with requires_grad (useful once layers are frozen for fine-tuning).</li></ul></details>

In [ ]:
# Your FPN on top of a pretrained ResNet-18: the shapes of the figure above
from torchvision.models.feature_extraction import create_feature_extractor
fpn_cls = FPN if tests.check_fpn_shapes(FPN, verbose=False) else day2.FPN
r18 = torchvision.models.resnet18(weights="IMAGENET1K_V1").eval()
body = create_feature_extractor(r18, {"layer2": "C3", "layer3": "C4", "layer4": "C5"})
fpn = fpn_cls([128, 256, 512], 256).eval()
with torch.no_grad():
    cs = body(torch.randn(1, 3, 256, 256))
    ps = fpn([cs["C3"], cs["C4"], cs["C5"]])
for (k, v), p_ in zip(cs.items(), ps):
    print(f"{k}: {tuple(v.shape)}  ->  P{k[1]}: {tuple(p_.shape)}")
print(f"FPN parameters: {torchkit.count_parameters(fpn):,d}")

What do the pyramid levels *see*? Our FPN is untrained, so we look at the trained FPN inside
torchvision's COCO RetinaNet. Below, the three main principal components of each level's 256
channels are shown as RGB: $P_3$ has fine detail (fur, edges, eyes), $P_5$ is coarse but each
of its cells summarises a whole head or body.

<details class="api-ref" data-api="torchvision.models.detection.retinanet_resnet50_fpn,np.linalg.svd"><summary><b>Functions used here</b> (click to open): <code>torchvision.models.detection.retinanet_resnet50_fpn</code> · <code>np.linalg.svd</code></summary>
<p><b><code>retinanet_resnet50_fpn(weights=None, weights_backbone=None, num_classes=None)</code></b><br>Builds torchvision&#x27;s RetinaNet: ResNet-50 + FPN (P3 to P7), classification and box heads shared by all levels, focal loss. Same interface as every torchvision detector: a list of (3, H, W) images in [0, 1]; eval mode returns one dict per image with &quot;boxes&quot; (xyxy pixels), &quot;labels&quot;, &quot;scores&quot;, train mode with targets a dict of losses. model.backbone(x) on a normalised (N, 3, H, W) batch returns just the pyramid.</p><pre>&gt;&gt;&gt; from torchvision.models.detection import retinanet_resnet50_fpn
&gt;&gt;&gt; m = retinanet_resnet50_fpn(weights=None, weights_backbone=None).eval()   # random, no download
&gt;&gt;&gt; feats = m.backbone(torch.zeros(1, 3, 128, 128))                          # the FPN levels P3 .. P7
&gt;&gt;&gt; {k: tuple(v.shape[1:]) for k, v in feats.items()}
{&#x27;0&#x27;: (256, 16, 16), &#x27;1&#x27;: (256, 8, 8), &#x27;2&#x27;: (256, 4, 4), &#x27;p6&#x27;: (256, 2, 2), &#x27;p7&#x27;: (256, 1, 1)}</pre><p><i>Watch out:</i> weights=RetinaNet_ResNet50_FPN_Weights.DEFAULT downloads the COCO weights; their .meta[&quot;categories&quot;] lists the 91 class names (index = label). Even with weights=None the default weights_backbone downloads ImageNet weights: pass weights_backbone=None for no download at all.</p><p>See also: <code>torchvision.models.detection.fcos_resnet50_fpn</code>, <code>torchvision.models.detection.fasterrcnn_resnet50_fpn_v2</code></p><p><b><code>U, S, Vt = np.linalg.svd(X, full_matrices=False)</code></b><br>Singular value decomposition: X = U @ np.diag(S) @ Vt, with S sorted from largest to smallest. If the columns of X (n, d) are centred (mean subtracted), the rows of Vt are the principal directions (PCA), and X @ Vt[:3].T projects every row onto the 3 most important ones.</p><pre>&gt;&gt;&gt; X = np.array([[2., 0.], [0., 1.], [-2., 0.], [0., -1.]])     # spread mostly along x
&gt;&gt;&gt; U, S, Vt = np.linalg.svd(X, full_matrices=False)
&gt;&gt;&gt; U.shape, S, Vt.shape
((4, 2), array([2.82842712, 1.41421356]), (2, 2))
&gt;&gt;&gt; np.allclose(U @ np.diag(S) @ Vt, X)
True</pre><p><i>Watch out:</i> full_matrices=False gives the small U of shape (n, k) instead of (n, n): with n = thousands of pixels the full one would need gigabytes.</p><p>See also: <code>torch.pca_lowrank</code>, <code>torch.linalg.svd</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torch.tensor,Tensor.numpy,Tensor.min,Tensor.max"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torch.tensor</code> · <code>Tensor.numpy</code> · <code>Tensor.min</code> · <code>Tensor.max</code></summary>
<ul><li><code>torch.tensor(data, dtype=None, device=None)</code> &nbsp; Builds a new tensor from a Python number, a (nested) list or a NumPy array, always copying the data. The dtype is inferred: integers give int64, floats give float32; pass dtype= to choose.</li><li><code>x.numpy()</code> &nbsp; Returns the tensor&#x27;s data as a NumPy array without copying: both share the same memory, so changing one changes the other. Used to hand results to Matplotlib or to NumPy code.</li><li><code>x.min() / x.min(dim)</code> &nbsp; x.min() is the smallest element (a 0-dimensional tensor); x.min(dim) works along one dimension and returns two tensors, the minimum values and their positions. NumPy arrays have the same method: arr.min() and arr.min(axis=...), which returns only the values.</li><li><code>x.max() / x.max(dim)</code> &nbsp; x.max() is the largest element (a 0-dimensional tensor); x.max(dim) works along one dimension and returns two tensors, the maximum values and their positions (indices). NumPy arrays have the same method: arr.max() and arr.max(axis=...), which returns only the values.</li></ul></details>

In [ ]:
from torchvision.models.detection import retinanet_resnet50_fpn, RetinaNet_ResNet50_FPN_Weights
retina = retinanet_resnet50_fpn(weights=RetinaNet_ResNet50_FPN_Weights.DEFAULT).eval()
pet_img = np.asarray(Image.open(pet_val.image_path(30)).convert("RGB").resize((384, 384))) / 255.0
xin = (torch.tensor(pet_img, dtype=torch.float32).permute(2, 0, 1) - torch.tensor(torchkit.IMAGENET_MEAN)[:, None, None]) / torch.tensor(torchkit.IMAGENET_STD)[:, None, None]
with torch.no_grad():
    feats = retina.backbone(xin[None])                       # OrderedDict '0'..'p7' = P3..P7
def pca_rgb(f):
    f = f[0].reshape(f.shape[1], -1).T.numpy(); f = f - f.mean(0)
    u, s_, vt = np.linalg.svd(f, full_matrices=False)
    z = f @ vt[:3].T; z = (z - z.min(0)) / (z.max(0) - z.min(0) + 1e-8)
    return z.reshape(int(np.sqrt(len(z))), -1, 3)
names = ["P3", "P4", "P5", "P6", "P7"]
fig, axes = plt.subplots(1, 6, figsize=(17, 3.3))
axes[0].imshow(pet_img); axes[0].set_title("input 384 x 384")
for ax, (k, f), n in zip(axes[1:], feats.items(), names):
    ax.imshow(pca_rgb(f), interpolation="nearest"); ax.set_title(f"{n}: {f.shape[-2]} x {f.shape[-1]} (stride {384 // f.shape[-1]})")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 4 · RetinaNet: one head shared by all levels

RetinaNet (Lin et al. 2017) = ResNet + FPN ($P_3$-$P_7$) + two small fully-convolutional heads
that run **with the same weights on every level** (an object is an object at every scale):

* classification tower: 4 x (3x3 conv 256 + ReLU), then a 3x3 conv with $A \cdot C$ outputs (sigmoid per class, no softmax);
* box tower: same structure, $A \cdot 4$ outputs (deltas vs anchors, as in D2-02);
* $A = 9$ anchors per location (3 scales x 3 ratios); anchor base size 32 on $P_3$ up to 512 on $P_7$.

One subtle detail matters a lot: the final classification bias is initialised to
$b = -\log\frac{1-\pi}{\pi}$ with $\pi = 0.01$, so that at the start *every* anchor predicts
"1% object". Without it, the ~100 000 background anchors produce a huge loss in the first
iterations and training diverges.

### Exercise 5 · `RetinaHead`

`RetinaHead(in_channels, num_anchors, num_classes, num_convs=4)`. Build two towers (`nn.Sequential`
of conv + ReLU), a `cls_logits` conv with `A * C` outputs and a `bbox_pred` conv with `A * 4`
outputs, initialise the classification bias to the prior, and make `forward(list of feature maps)`
return `(list of class maps, list of box maps)`: for an input level `(N, 256, H, W)` the class map
is `(N, A * C, H, W)` and the box map `(N, A * 4, H, W)`, one entry per level.

<details class="api-ref" data-api="nn.init.constant_"><summary><b>Functions you will need</b> (click to open): <code>nn.init.constant_</code></summary>
<p><b><code>nn.init.constant_(tensor, value)</code></b><br>Fills a parameter with one value, in place (the trailing _). Detectors set the bias of the last classification conv to -log((1 - pi) / pi) with pi = 0.01, so that at the start every anchor predicts &quot;1% object&quot; and the many background anchors do not blow up the first losses.</p><pre>&gt;&gt;&gt; import math
&gt;&gt;&gt; conv = nn.Conv2d(256, 3, 3, padding=1)
&gt;&gt;&gt; _ = nn.init.constant_(conv.bias, -math.log((1 - 0.01) / 0.01))
&gt;&gt;&gt; conv.bias.detach()
tensor([-4.5951, -4.5951, -4.5951])
&gt;&gt;&gt; torch.sigmoid(conv.bias.detach())               # the prior probability
tensor([0.0100, 0.0100, 0.0100])</pre><p>See also: <code>nn.init.normal_</code>, <code>nn.init.zeros_</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="nn.Sequential,nn.ReLU,torch.zeros,torch.sigmoid"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>nn.Sequential</code> · <code>nn.ReLU</code> · <code>torch.zeros</code> · <code>torch.sigmoid</code></summary>
<ul><li><code>nn.Sequential(layer1, layer2, ...)</code> &nbsp; A model made of layers applied one after the other, the output of each being the input of the next. Saves writing a class when forward() would only chain layers. The layers can be read back by position: model[0].</li><li><code>nn.ReLU()</code> &nbsp; The ReLU activation as a layer (a module): max(x, 0) element by element, no parameters. Used inside nn.Sequential; in a forward() you can write F.relu(x) instead.</li><li><code>torch.zeros(*size, dtype=None, device=None)</code> &nbsp; A new tensor of the given shape filled with 0 (float32 unless dtype says otherwise). Used to allocate outputs and to build test inputs of a given shape.</li><li><code>torch.sigmoid(x)</code> &nbsp; Squashes every value (a logit) into (0, 1): sigmoid(x) = 1 / (1 + exp(-x)). Turns an objectness logit into a probability; D2-03 and D2-04 also use it to keep predicted box coordinates between 0 and 1.</li></ul></details>

In [ ]:
import math
class RetinaHead(nn.Module):
    '''RetinaHead(C_in, A, K): forward(list of feature maps (N, C_in, H, W), one per level) ->
    (list of class maps (N, A * K, H, W), list of box maps (N, A * 4, H, W)); the same weights on every level.'''
    def __init__(self, in_channels, num_anchors, num_classes, num_convs=4, prior=0.01):
        super().__init__()
        # 1. the classification tower: num_convs times (3x3 conv in_channels -> in_channels, padding 1, then ReLU)
        cls_layers = []
        for _ in range(num_convs):
            cls_layers.append(nn.Conv2d(in_channels, in_channels, 3, padding=1))
            cls_layers.append(nn.ReLU())
        self.cls_tower = nn.Sequential(*cls_layers)      # runs the layers of the list one after the other
        # 2. the box tower: the same structure, its own weights
        box_layers = []
        for _ in range(num_convs):
            box_layers.append(nn.Conv2d(in_channels, in_channels, 3, padding=1))
            box_layers.append(nn.ReLU())
        self.box_tower = nn.Sequential(*box_layers)
        # 3. A * K class logits per location
        self.cls_logits = nn.Conv2d(in_channels, num_anchors * num_classes, 3, padding=1)
        # 4. A * 4 box deltas per location
        self.bbox_pred = nn.Conv2d(in_channels, num_anchors * 4, 3, padding=1)
        # 5. the prior: every anchor starts at P(object) = prior
        bias_value = -math.log((1 - prior) / prior)      # sigmoid(bias_value) = prior
        nn.init.constant_(self.cls_logits.bias, bias_value)

    def forward(self, feats):
        cls_maps = []
        box_maps = []
        for f in feats:                                  # f: (N, C_in, H, W), one pyramid level
            # 6. classification branch
            cls_features = self.cls_tower(f)             # (N, C_in, H, W)
            cls_maps.append(self.cls_logits(cls_features))   # (N, A * K, H, W)
            # 7. box branch
            box_features = self.box_tower(f)             # (N, C_in, H, W)
            box_maps.append(self.bbox_pred(box_features))    # (N, A * 4, H, W)
        return cls_maps, box_maps

tests.check_retina_head(RetinaHead)

In [ ]:
# How many predictions does RetinaNet make on a 512 x 512 image? (A = 9, C = 80 COCO classes)
head = day2.RetinaHead(256, 9, 80)
levels = {}                                      # level name -> grid size on a 512 px image (stride 2 ** l)
for l in range(3, 8):
    levels[f"P{l}"] = 512 // 2 ** l
level_inputs = []                                # one empty feature map (1, 256, s, s) per level
for s in levels.values():
    level_inputs.append(torch.zeros(1, 256, s, s))
with torch.no_grad():
    cls_out, box_out = head(level_inputs)
tot = 0
print(f"{'level':6}{'grid':>9}{'cls output':>22}{'anchors':>10}")
for (n, s), c in zip(levels.items(), cls_out):
    tot += s * s * 9
    print(f"{n:6}{f'{s} x {s}':>9}{str(tuple(c.shape[1:])):>22}{s * s * 9:>10,d}")
print(f"{'total':6}{'':>9}{'':>22}{tot:>10,d}  anchors, x 80 class scores each")
print(f"head parameters (shared by all 5 levels): {torchkit.count_parameters(head) / 1e6:.2f} M; "
      f"initial P(object) = {torch.sigmoid(cls_out[0]).mean():.3f}")

## 5 · Class imbalance and focal loss

RetinaNet scores ~50 000 anchors per image; maybe 20 are positive. With plain binary
cross-entropy, $\mathrm{CE}(p_t) = -\log p_t$ where $p_t$ is the predicted probability of the
*true* label, each easy negative (say $p_t = 0.99$) contributes only $0.01$ - but there are
50 000 of them, and together they swamp the gradient of the 20 positives. Two-stage detectors
avoid this by sampling (D2-02); SSD by hard-negative mining (D2-01). **Focal loss** keeps all
anchors and down-weights the easy ones with a modulating factor:

$$\mathrm{FL}(p_t) = -\,\alpha_t\,(1 - p_t)^{\gamma}\,\log p_t .$$

* $\gamma = 0$ gives (alpha-weighted) cross-entropy; $\gamma = 2$ (the paper's default) makes an example with $p_t = 0.9$ count **100x less** and one with $p_t = 0.99$ **10 000x less**;
* $\alpha_t = \alpha$ for positives, $1-\alpha$ for negatives ($\alpha = 0.25$): a small extra re-balancing (it *down*-weights positives, because $\gamma$ already removed most negatives).

<details class="api-ref" data-api="np.concatenate,np.log,np.clip"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.concatenate</code> · <code>np.log</code> · <code>np.clip</code></summary>
<ul><li><code>np.concatenate(arrays, axis=0)</code> &nbsp; Joins arrays end to end along an EXISTING axis; all other dimensions must match. axis=0 appends rows; axis=1 puts columns side by side, e.g. top-left corners (N, 2) and bottom-right corners (N, 2) into xyxy boxes (N, 4). The AP code uses it to pad a curve with one value at each end.</li><li><code>np.log(x)</code> &nbsp; The natural logarithm (base e) of every element. ln C is the cross-entropy loss of a classifier that guesses uniformly among C classes: ln 37 = 3.61.</li><li><code>np.clip(a, a_min, a_max)</code> &nbsp; Limits every element to [a_min, a_max]: smaller values become a_min, larger ones a_max. Keeps simulated scores inside [0, 1], and probabilities away from 0 before a log.</li></ul></details>

In [ ]:
pt = np.linspace(0.001, 1, 500)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6))
for g, c in zip([0, 0.5, 1, 2, 5], [GREY, "#9ecae1", BLUE, RED, "#6a3d9a"]):
    axes[0].plot(pt, -(1 - pt) ** g * np.log(pt), color=c, lw=2.5 if g == 2 else 1.8, label=f"gamma = {g}" + (" (CE)" if g == 0 else ""))
axes[0].axvspan(0.6, 1, color=GREEN, alpha=0.08); axes[0].text(0.75, 3.6, "well-classified\nexamples", ha="center", color=GREEN)
axes[0].set_xlabel("p_t (probability of the true class)"); axes[0].set_ylabel("loss"); axes[0].set_ylim(0, 5); axes[0].legend(); axes[0].grid(alpha=.3)
axes[0].set_title("focal loss for several gamma (alpha = 1)")
# contribution of easy negatives on a simulated image: 50 000 anchors
rng = np.random.default_rng(0)
n_easy, n_hard, n_pos = 49_000, 980, 20
p_neg = np.concatenate([rng.beta(1, 60, n_easy), rng.beta(3, 4, n_hard)])     # predicted P(object) for negatives
p_pos = rng.beta(4, 3, n_pos)
def fl(pt_, g, a):
    return -a * (1 - pt_) ** g * np.log(np.clip(pt_, 1e-7, 1))
rows = []
for name, g, a in [("CE", 0, 0.5), ("focal (gamma=2, alpha=0.25)", 2, 0.25)]:
    easy = fl(1 - p_neg[:n_easy], g, 1 - a).sum()           # total loss of the easy negatives (p_t = 1 - p)
    hard = fl(1 - p_neg[n_easy:], g, 1 - a).sum()           # of the hard negatives
    pos = fl(p_pos, g, a).sum()                             # of the positives (p_t = p)
    rows.append((name, easy, hard, pos))
x = np.arange(2)
tot = np.array([r[1] + r[2] + r[3] for r in rows])
for k, (lab, c) in enumerate(zip(["49 000 easy negatives", "980 hard negatives", "20 positives"], [GREY, ORANGE, GREEN])):
    vals = np.array([r[1 + k] for r in rows]) / tot
    bottom = np.array([sum(r[1 + j] for j in range(k)) for r in rows]) / tot
    axes[1].bar(x, vals, bottom=bottom, color=c, label=lab, width=0.5)
    for xi, v, b in zip(x, vals, bottom):
        if v > 0.04: axes[1].text(xi, b + v / 2, f"{100 * v:.0f}%", ha="center", va="center", color="white", weight="bold")
axes[1].set_xticks(x); axes[1].set_xticklabels([r[0] for r in rows]); axes[1].set_ylabel("share of the total loss")
axes[1].legend(loc="upper left", bbox_to_anchor=(1.0, 1.0)); axes[1].set_title("who drives the gradient? (simulated image, 50 000 anchors)")
plt.tight_layout(); plt.show()

With cross-entropy the 49 000 easy negatives - each individually negligible - make up more than
half of the loss. With focal loss their share drops below 1%: the loss now comes from the examples
the network still gets wrong (here mostly hard negatives; the 20 positives are few, and $\alpha = 0.25$
deliberately down-weights them because $\gamma$ has already removed the competition). Compared with
hard-negative mining / OHEM (online hard example mining: keep only the highest-loss examples of
each batch), nothing is thrown away and there is no sorting step; the re-weighting is smooth.

### Lab · Focal loss
**Predict first:** 100 000 easy negatives at $p_t = 0.99$ against 20 positives at $p_t = 0.3$.
Under plain cross-entropy, roughly what share of the *total* loss comes from the easy
negatives? And with $\gamma = 2$? Check in the right half of the lab, then work through the
"Try this" list (5 minutes).

In [ ]:
labs.show("focal-loss")

### Exercise 6 · focal loss in PyTorch

`focal_loss(logits, targets, gamma=2.0, alpha=0.25)` -> mean over all elements (a 0-d tensor).
`logits` and `targets` have the same shape; `targets` holds 0. / 1. floats. Compute the
log term with `F.binary_cross_entropy_with_logits(..., reduction="none")` (stable for large
logits), $p = \sigma(\text{logits})$, $p_t = p\,t + (1-p)(1-t)$, $\alpha_t = \alpha t + (1-\alpha)(1-t)$.
The checker compares with `torchvision.ops.sigmoid_focal_loss`. For a single positive with
$p_t = 0.9$ your function returns the lab's FL for that marker, 2.63e-4.

In [ ]:
def focal_loss(logits, targets, gamma=2.0, alpha=0.25):
    '''logits, targets: tensors of the same shape (targets hold 0. or 1.) -> the mean focal loss, a 0-d tensor.'''
    # 1. ce, the log term of every element (-log p_t, computed stably)
    ce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    # 2. p, the predicted probability of "object"
    p = torch.sigmoid(logits)
    # 3. p_t, the probability of the true label
    p_t = p * targets + (1 - p) * (1 - targets)
    # 4. alpha_t: alpha for positives, 1 - alpha for negatives
    alpha_t = alpha * targets + (1 - alpha) * (1 - targets)
    # 5. the focal loss of every element: (1 - p_t) ** gamma shrinks the easy ones (p_t close to 1)
    loss = alpha_t * (1 - p_t) ** gamma * ce
    # 6. its mean
    return loss.mean()

tests.check_focal_loss_torch(focal_loss)

## 6 · Anchor-free heads: FCOS, CenterNet, modern YOLO

Anchors bring hyper-parameters (scales, ratios, IoU thresholds) that must be tuned per dataset.
Anchor-free detectors predict boxes from **points**:

* **FCOS** (Tian et al. 2019): every feature-map location inside a GT box is a positive and
  regresses the four distances $(l, t, r, b)$ to the box sides (FPN levels are assigned by box
  size). Locations near the box border produce poor boxes, so FCOS adds a **centre-ness**
  branch, $\sqrt{\frac{\min(l,r)}{\max(l,r)} \cdot \frac{\min(t,b)}{\max(t,b)}}$, multiplied into the
  score at test time so that NMS keeps the central predictions.
* **CenterNet** ("Objects as points", Zhou et al. 2019): predict a **heat map** per class with a
  Gaussian peak at each object centre, plus the box size at the peak. Decoding = local maxima
  (a 3 x 3 max-pool comparison) - no NMS at all.
* **Modern YOLO heads** (YOLOX, v8, 11): anchor-free, **decoupled** (separate classification and
  box branches, like RetinaNet's towers), box sides predicted as a *distribution* over bins
  (**Distribution Focal Loss**) instead of a single number, and dynamic label assignment
  (positives chosen by a cost that mixes classification score and IoU).

### Exercise 7 · FCOS centre-ness

`centerness(ltrb)` for an array `(..., 4)` of positive distances.

<details class="api-ref" data-api="np.minimum,np.maximum"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.minimum</code> · <code>np.maximum</code></summary>
<ul><li><code>np.minimum(a, b)</code> &nbsp; Element-wise minimum of two arrays (broadcast against each other): out[i] = min(a[i], b[i]). For two xyxy boxes, the bottom-right corner of their intersection is np.minimum of the two bottom-right corners, and the top-left corner is np.maximum of the two top-left corners.</li><li><code>np.maximum(a, b)</code> &nbsp; The element-wise maximum of two arrays, or of an array and a number. np.maximum(values, 1e-30) replaces values below 1e-30 (including 0) so a log-scale plot does not break on log(0). Not to be confused with np.max, which returns the single largest value.</li></ul></details>

In [ ]:
def centerness(ltrb):
    '''ltrb (..., 4): positive distances (l, t, r, b) from a point to the left, top, right and bottom box sides
    -> centre-ness (...), in [0, 1]: 1 at the box centre, close to 0 near a side.'''
    ltrb = np.asarray(ltrb, float)
    # 1. the four distances, one array each: shape (...)
    l = ltrb[..., 0]
    t = ltrb[..., 1]
    r = ltrb[..., 2]
    b = ltrb[..., 3]
    # 2. horizontal: 1 when the point is half-way between left and right
    horizontal = np.minimum(l, r) / np.maximum(l, r)
    # 3. vertical: the same for top and bottom
    vertical = np.minimum(t, b) / np.maximum(t, b)
    # 4. the square root of their product
    return np.sqrt(horizontal * vertical)

tests.check_centerness(centerness)

<details class="api-ref" data-api="np.mgrid,F.max_pool2d,np.ma.masked_where"><summary><b>Functions used here</b> (click to open): <code>np.mgrid</code> · <code>F.max_pool2d</code> · <code>np.ma.masked_where</code></summary>
<p><b><code>yy, xx = np.mgrid[0:H, 0:W]</code></b><br>Not a function call but indexing with slices: gives the row index and the column index of every position of an H x W grid as two (H, W) arrays (like np.meshgrid with indexing=&quot;ij&quot;). Adding 0.5 gives pixel centres.</p><pre>&gt;&gt;&gt; yy, xx = np.mgrid[0:2, 0:3]
&gt;&gt;&gt; yy
array([[0, 0, 0],
       [1, 1, 1]])
&gt;&gt;&gt; xx
array([[0, 1, 2],
       [0, 1, 2]])</pre><p>See also: <code>np.meshgrid</code>, <code>torch.meshgrid</code></p><p><b><code>F.max_pool2d(x, kernel_size, stride=None, padding=0)</code></b><br>The function form of nn.MaxPool2d: each kernel_size x kernel_size window of an (N, C, H, W) tensor is replaced by its maximum (stride defaults to kernel_size). With kernel 3, stride 1, padding 1 the size is kept and each value becomes the max of its 3 x 3 neighbourhood, so x == F.max_pool2d(x, 3, 1, 1) marks the local maxima: CenterNet&#x27;s peak finding without NMS.</p><pre>&gt;&gt;&gt; x = torch.arange(16.).reshape(1, 1, 4, 4)
&gt;&gt;&gt; F.max_pool2d(x, 2)[0, 0]                          # 2 x 2 windows: H and W halved
tensor([[ 5.,  7.],
        [13., 15.]])
&gt;&gt;&gt; h = torch.tensor([[0., 1., 0.], [0., 0., 0.], [0., 0., 2.]])[None, None]
&gt;&gt;&gt; ((h == F.max_pool2d(h, 3, 1, 1)) &amp; (h &gt; 0))[0, 0]    # local maxima
tensor([[False,  True, False],
        [False, False, False],
        [False, False,  True]])</pre><p>See also: <code>nn.MaxPool2d</code></p><p><b><code>np.ma.masked_where(condition, a)</code></b><br>Returns a masked array: a copy of a in which the elements where condition is True are marked as missing. Matplotlib draws masked pixels transparent, so imshow(np.ma.masked_where(m == 0, m)) overlays only the non-zero part of a map on an image.</p><pre>&gt;&gt;&gt; m = np.array([0.0, 0.5, 0.0, 1.0])
&gt;&gt;&gt; np.ma.masked_where(m == 0, m)
masked_array(data=[--, 0.5, --, 1.0],
             mask=[ True, False,  True, False],
       fill_value=1e+20)</pre><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="np.stack,Tensor.ravel,np.exp,np.where"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>np.stack</code> · <code>Tensor.ravel</code> · <code>np.exp</code> · <code>np.where</code></summary>
<ul><li><code>np.stack(arrays, axis=0)</code> &nbsp; Joins arrays of the same shape along a NEW axis: k arrays of shape (N,) give (k, N) with axis=0, or (N, k) with axis=-1. The box conversions use it to put four (N,) coordinate columns back together into one (N, 4) array.</li><li><code>x.ravel()</code> &nbsp; Returns all values as one flat 1-D tensor, row after row: the same as x.flatten() or x.reshape(-1). NumPy arrays have the same method; D1-02 calls it on an array (after .numpy()) to get a flat list of values for a histogram.</li><li><code>np.exp(x)</code> &nbsp; e = 2.718... to the power of every element. Soft-NMS multiplies scores by exp(-IoU^2 / sigma): 1 for no overlap, close to 0 for a heavy duplicate. It also decodes log-encoded box sizes (D2-02).</li><li><code>np.where(condition)  or  np.where(condition, x, y)</code> &nbsp; With one argument: the indices where condition is True, as a tuple with one index array per dimension (take [0] for a 1-D array). With three: a new array that takes x where condition is True and y elsewhere, element by element.</li></ul></details>

In [ ]:
fc_img, fc_boxes, fc_labels, _ = shapes.make_sample(np.random.default_rng(7), 128)
box = fc_boxes[np.argmax(boxes.box_area(fc_boxes))]
yy, xx = np.mgrid[0:128, 0:128] + 0.5
pts = np.stack([xx.ravel(), yy.ravel()], 1)
ltrb = day2.fcos_ltrb(pts, box)
cn = day2.centerness(ltrb).reshape(128, 128)

# CenterNet-style heat map: a Gaussian at every centre (sigma ~ box size / 6), then peaks by 3x3 max-pool
heat = np.zeros((128, 128))
for b in fc_boxes:
    cx, cy = (b[0] + b[2]) / 2, (b[1] + b[3]) / 2; sig = (b[2] - b[0]) / 6
    heat = np.maximum(heat, np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sig ** 2)))
ht = torch.tensor(heat)[None, None]
peaks = (ht == F.max_pool2d(ht, 3, 1, 1)) & (ht > 0.5)
py, px = np.where(peaks[0, 0].numpy())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(fc_img); p = np.array([box[0] + 0.3 * (box[2] - box[0]), box[1] + 0.6 * (box[3] - box[1])])
l, t, r, b = day2.fcos_ltrb(p[None], box)[0]
for (dx, dy), lab_, v in [((-l, 0), "l", l), ((0, -t), "t", t), ((r, 0), "r", r), ((0, b), "b", b)]:
    axes[0].annotate("", (p[0] + dx, p[1] + dy), p, arrowprops=dict(arrowstyle="->", lw=2, color="k"))
    axes[0].text(p[0] + dx / 2 + 1, p[1] + dy / 2 - 1, f"{lab_}={v:.0f}", fontsize=9, weight="bold")
axes[0].plot(*p, "o", color=RED); viz.draw_boxes(axes[0], box, edgecolor="k", lw=2)
axes[0].set_title("FCOS: a point regresses (l, t, r, b)")
axes[1].imshow(fc_img); axes[1].imshow(np.ma.masked_where(cn == 0, cn), cmap="magma", alpha=0.8, vmin=0, vmax=1)
axes[1].set_title("centre-ness of every point inside the box")
axes[2].imshow(fc_img); axes[2].imshow(heat, cmap="jet", alpha=0.5)
axes[2].plot(px, py, "w*", ms=14, mec="k"); axes[2].set_title(f"CenterNet heat map: {len(px)} peaks = {len(fc_boxes)} objects")
for ax in axes: ax.set_xlim(0, 128); ax.set_ylim(128, 0); ax.axis("off")
plt.tight_layout(); plt.show()

What to look at: centre-ness reaches 1 only at the box centre ($l = r$ and $t = b$), stays high
along the two centre lines (the bright cross) and fades to 0 at the border, so predictions from
border points are down-weighted at test time.
The CenterNet map has exactly one local maximum per object: the 3 x 3 max-pool test finds them
without any NMS.

## 7 · Library detectors on Pet images (no training)

torchvision ships RetinaNet and FCOS trained on COCO, which has `cat` and `dog` classes. We run
both on a few Pet validation images at 512 px. Note what they detect: **whole animals**, while
our Pet labels are **heads** - the model is only as good as the match between its training labels
and your task. That mismatch is why we fine-tune (D2-04).

<details class="api-ref" data-api="torchvision.models.detection.fcos_resnet50_fpn"><summary><b>Functions used here</b> (click to open): <code>torchvision.models.detection.fcos_resnet50_fpn</code></summary>
<p><b><code>fcos_resnet50_fpn(weights=None, weights_backbone=None, num_classes=None, min_size=800, max_size=1333)</code></b><br>Builds torchvision&#x27;s FCOS: anchor-free (one prediction per feature-map location, so num_anchors = 1), ResNet-50 + FPN, focal loss and centre-ness. Same interface as the other torchvision detectors: a list of (3, H, W) images in [0, 1]; eval mode returns per image a dict with &quot;boxes&quot;, &quot;labels&quot;, &quot;scores&quot;, train mode with targets a dict of losses. Its class layer is model.head.classification_head.cls_logits.</p><pre>&gt;&gt;&gt; from torchvision.models.detection import fcos_resnet50_fpn
&gt;&gt;&gt; m = fcos_resnet50_fpn(weights=None, weights_backbone=None)            # random, no download
&gt;&gt;&gt; head = m.head.classification_head
&gt;&gt;&gt; head.num_anchors, head.num_classes, head.cls_logits.out_channels       # 91 COCO classes
(1, 91, 91)
&gt;&gt;&gt; m.transform.min_size, m.transform.max_size                             # inputs are resized to this
((800,), 1333)</pre><p><i>Watch out:</i> weights=&quot;DEFAULT&quot; (FCOS_ResNet50_FPN_Weights.DEFAULT) downloads the COCO weights; the default weights_backbone downloads ImageNet weights even with weights=None, so pass weights_backbone=None too for no download at all.</p><p>See also: <code>torchvision.models.detection.retinanet_resnet50_fpn</code>, <code>nn.init.normal_</code>, <code>nn.init.constant_</code></p><p><small>Any of these again later: <code>apiref.show("name")</code>.</small></p></details>

<details class="api-ref" data-api="torchvision.transforms.v2.Compose,torchvision.transforms.v2.ToImage,torchvision.transforms.v2.ToDtype"><summary><b>Functions from earlier notebooks, used again here</b> (click to open): <code>torchvision.transforms.v2.Compose</code> · <code>torchvision.transforms.v2.ToImage</code> · <code>torchvision.transforms.v2.ToDtype</code></summary>
<ul><li><code>v2.Compose([transform1, transform2, ...])</code> &nbsp; Chains transforms into one: calling the result applies them in the listed order, each on the output of the previous one. A whole per-image pipeline (resize, augment, convert to float, normalise) becomes one callable that a Dataset applies to every sample.</li><li><code>v2.ToImage()</code> &nbsp; Transform that turns a PIL image or an (H, W, C) NumPy array into an image tensor of shape (C, H, W). Values and dtype are kept (uint8 stays 0..255), so it is almost always followed by v2.ToDtype(torch.float32, scale=True) to get floats in [0, 1].</li><li><code>v2.ToDtype(torch.float32, scale=True)</code> &nbsp; Converts the image to another dtype. With scale=True it also rescales the values to the range of the new type: uint8 0 .. 255 becomes float 0.0 .. 1.0. The step between the cheap uint8 transforms and Normalize.</li></ul></details>

In [ ]:
from torchvision.models.detection import fcos_resnet50_fpn, FCOS_ResNet50_FPN_Weights
cats = RetinaNet_ResNet50_FPN_Weights.DEFAULT.meta["categories"]
fcos = fcos_resnet50_fpn(weights=FCOS_ResNet50_FPN_Weights.DEFAULT).eval()
for m in (retina, fcos):
    m.transform.min_size, m.transform.max_size = (512,), 683
from torchvision.transforms import v2
tf = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
idx = [5, 40, 77, 120, 260]
imgs = []
for i in idx:
    pil_img = Image.open(pet_val.image_path(i)).convert("RGB")
    imgs.append(tf(pil_img))                     # (3, H, W) float in [0, 1]
res = {}                                         # model name -> one output dict per image
for name, m in [("RetinaNet", retina), ("FCOS", fcos)]:
    t0 = time.time()
    res[name] = []
    with torch.no_grad():
        for im in imgs:
            outputs = m([im])                    # torchvision detectors take a list of images ...
            res[name].append(outputs[0])         # ... and return a list of dicts: boxes, scores, labels
    print(f"{name}: {(time.time() - t0) / len(imgs):.2f} s / image on CPU at 512 px")
fig, axes = plt.subplots(2, len(idx), figsize=(17, 7))
for r, name in enumerate(res):
    for c, (im, out) in enumerate(zip(imgs, res[name])):
        ax = axes[r, c]; ax.imshow(im.permute(1, 2, 0)); ax.axis("off")
        k = out["scores"] > 0.5
        viz.draw_boxes(ax, out["boxes"][k].numpy(), labels=[cats[l] for l in out["labels"][k]], scores=out["scores"][k].numpy(), edgecolor=ORANGE, lw=2)
        _, tgt = pet_val[idx[c]]
        sx, sy = im.shape[2] / 224, im.shape[1] / 224
        viz.draw_boxes(ax, tgt["boxes"].numpy() * [sx, sy, sx, sy], edgecolor=GREEN, lw=2)
        if c == 0: ax.set_title(f"{name} (COCO weights)", loc="left", fontsize=11)
fig.suptitle("orange: COCO detections with score > 0.5   green: the Pet *head* annotation", y=1.0)
plt.tight_layout(); plt.show()

## 8 · Summary

* A one-stage detector is a convnet whose last layer outputs $A(1 + 4 + C)$ channels per cell; the
  target tensor says which cell / anchor is responsible for which object.
* Our mini detector (300 k parameters, 2 minutes of CPU) reaches mAP@0.5 > 0.95 on shapes with
  BCE (objectness, all cells) + L1 (box, positives) + CE (class, positives), decode and NMS.
* Multi-scale is essential: SSD predicts from several maps; FPN adds a top-down pathway so that
  every level has both resolution and semantics. Heads are shared across levels.
* Dense detectors face ~1000:1 background imbalance. Focal loss $-\alpha_t(1-p_t)^\gamma\log p_t$
  silences easy negatives; the prior-probability bias init keeps the first iterations stable.
* Anchor-free heads (FCOS distances + centre-ness, CenterNet peaks, YOLOv8's decoupled DFL head)
  removed anchor hyper-parameters and are today's default.

### Self-check
1. How many output channels does a head need for 3 anchors per cell and 20 classes (YOLOv3-style, with objectness)?
2. Why does the box loss only use the responsible cells, while the objectness loss uses all cells?
3. In an FPN, why add the upsampled coarse map instead of just using $C_3$ for small objects?
4. With $\gamma = 2$, by how much is an example with $p_t = 0.9$ down-weighted relative to CE?
5. What would happen at the first iteration of RetinaNet if the classification bias were initialised to 0?
6. Which FCOS predictions does centre-ness suppress, and why does that help NMS?

### Go deeper
* Redmon et al., *You Only Look Once: Unified, Real-Time Object Detection* (YOLOv1, CVPR 2016).
* Liu et al., *SSD: Single Shot MultiBox Detector* (ECCV 2016).
* Lin et al., *Feature Pyramid Networks for Object Detection* (CVPR 2017).
* Lin et al., *Focal Loss for Dense Object Detection* (RetinaNet, ICCV 2017).
* Tian et al., *FCOS: Fully Convolutional One-Stage Object Detection* (ICCV 2019).
* Zhou et al., *Objects as Points* (CenterNet, 2019).